# 🚬 Notebook 3: Cigarette/Vape Detection — YOLOv8 + CBAM
**Cairo University | FCAI | AI Department | 2025-2026**
- Classes: 4 (smoke, person, cigarette, vape)
- Dataset: Smoking_person v3 (Roboflow) — 5,658 images
- Contribution: YOLOv8-det + CBAM in backbone

In [ ]:
!pip install ultralytics roboflow -q
import torch; print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Download Dataset ───────────────────────────────────────────────────────
from roboflow import Roboflow
rf      = Roboflow(api_key="5tsw5DCNIq4ttW5TNmOP")
project = rf.workspace("visionwork").project("smoking_person")
version = project.version(3)
dataset = version.download("yolov8")
print(f'✅ Dataset: {dataset.location}')

In [ ]:
# ── Check Balance ──────────────────────────────────────────────────────────
import os
from collections import Counter
import matplotlib.pyplot as plt

class_names = ['Smoke', 'Person', 'Cigarette', 'Vape']
counts = Counter()

label_dir = f'{dataset.location}/train/labels'
for lf in os.listdir(label_dir):
    if lf.endswith('.txt'):
        with open(os.path.join(label_dir, lf)) as f:
            for line in f:
                cid = int(line.split()[0])
                if cid < len(class_names): counts[class_names[cid]] += 1

print('📊 CLASS BALANCE CHECK (Training Set):')
total = sum(counts.values())
for cls, cnt in counts.most_common():
    bar = '█' * int(cnt/total*50)
    print(f'   {cls:<12}: {cnt:4d} ({cnt/total*100:.1f}%) {bar}')

# Identify imbalance
max_c = max(counts.values())
min_c = min(counts.values())
print(f'\n   Balance ratio: {min_c/max_c:.3f} (1.0 = perfect)')
if min_c/max_c < 0.3:
    print('   ⚠️ Severe class imbalance detected (Vape class)')
    print('   → We handle this with class_weights in training')

plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values(), color=['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4'])
plt.title('Class Distribution — Smoking Dataset (Training Set)')
plt.ylabel('Instance Count')
for i, (cls, cnt) in enumerate(counts.items()):
    plt.text(i, cnt+5, str(cnt), ha='center', fontweight='bold')
plt.savefig('smoking_balance.png', dpi=150)
plt.show()

In [ ]:
# ── CBAM + Register ────────────────────────────────────────────────────────
import torch, torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.ap = nn.AdaptiveAvgPool2d(1); self.mp = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(nn.Conv2d(c,c//r,1,bias=False),nn.ReLU(),nn.Conv2d(c//r,c,1,bias=False))
        self.sg = nn.Sigmoid()
    def forward(self, x): return self.sg(self.fc(self.ap(x))+self.fc(self.mp(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.cv = nn.Conv2d(2,1,k,padding=k//2,bias=False); self.sg = nn.Sigmoid()
    def forward(self, x):
        return self.sg(self.cv(torch.cat([torch.mean(x,1,True),torch.max(x,1,True)[0]],1)))

class CBAM(nn.Module):
    """CBAM — Woo et al. ECCV 2018 | Our Contribution"""
    def __init__(self, c, r=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(c,r); self.sa = SpatialAttention(k)
    def forward(self, x): return x*self.sa(x*self.ca(x))

from ultralytics.nn import modules; modules.CBAM = CBAM
import ultralytics.nn.tasks as t; t.CBAM = CBAM
print('✅ CBAM registered!')

In [ ]:
# ── Create CBAM YAML ───────────────────────────────────────────────────────
yaml_str = """
# YOLOv8n + CBAM for Smoking Detection
# Contribution: Cairo University | FCAI | AI Dept | 2026
nc: 4
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, CBAM, [1024]]
  - [-1, 1, SPPF, [1024, 5]]
head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[16, 19, 22], 1, Detect, [nc]]
"""
with open('yolov8n_cbam.yaml', 'w') as f: f.write(yaml_str)
print('✅ YAML saved!')

In [ ]:
# ── Train Baseline ─────────────────────────────────────────────────────────
from ultralytics import YOLO
m_base = YOLO('yolov8n.pt')
r_base = m_base.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50, imgsz=640, batch=16,
    name='smoking_baseline', project='runs/smoking',
    device=0, optimizer='SGD', lr0=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3, freeze=10,
    patience=10, save=True, plots=True, verbose=False
)
base_map = r_base.results_dict.get('metrics/mAP50(B)', 0)
print(f'✅ Baseline mAP: {base_map*100:.1f}%')

In [ ]:
# ── Train CBAM ─────────────────────────────────────────────────────────────
from ultralytics import YOLO
m_cbam = YOLO('yolov8n_cbam.yaml')
r_cbam = m_cbam.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50, imgsz=640, batch=16,
    name='smoking_cbam', project='runs/smoking',
    device=0, optimizer='SGD', lr0=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3,
    patience=10, save=True, plots=True, verbose=False
)
cbam_map = r_cbam.results_dict.get('metrics/mAP50(B)', 0)
print(f'✅ CBAM mAP: {cbam_map*100:.1f}%')
print(f'   Improvement: {(cbam_map-base_map)*100:+.2f}%')

In [ ]:
# ── Compare & Download ─────────────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt, numpy as np, shutil, os
from google.colab import files

def find_csv(kw):
    for root,d,fs in os.walk('runs/smoking'):
        if kw in root:
            for f in fs:
                if f=='results.csv': return os.path.join(root,f)

df_b = pd.read_csv(find_csv('baseline')); df_b.columns = df_b.columns.str.strip()
df_c = pd.read_csv(find_csv('cbam'));     df_c.columns = df_c.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Smoking: YOLOv8 vs YOLOv8+CBAM\nCairo University 2026', fontsize=12, fontweight='bold')
map_col = [c for c in df_b.columns if 'mAP50' in c and 'B' in c]
if map_col:
    axes[0].plot(df_b[map_col[0]], label='Baseline', color='#2196F3', linewidth=2)
    axes[0].plot(df_c[map_col[0]], label='CBAM', color='#FF5722', linewidth=2, linestyle='--')
    axes[0].set_title('mAP@0.5'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

x = ['mAP@0.5']
axes[1].bar([0-0.2], [base_map*100], 0.35, label='Baseline', color='#2196F3')
axes[1].bar([0+0.2], [cbam_map*100], 0.35, label='CBAM', color='#FF5722')
axes[1].set_xticks([0]); axes[1].set_xticklabels(x)
axes[1].set_title('Final mAP Comparison'); axes[1].legend(); axes[1].set_ylim(0,100)
plt.savefig('smoking_comparison.png', dpi=150)
plt.show()

print('='*55)
print(f'  Baseline mAP : {base_map*100:.1f}%')
print(f'  CBAM mAP     : {cbam_map*100:.1f}%')
print(f'  Improvement  : {(cbam_map-base_map)*100:+.2f}%')
print('='*55)

for root,d,fs in os.walk('runs/smoking/smoking_cbam'):
    for f in fs:
        if f=='best.pt': shutil.copy(os.path.join(root,f),'smoking_cbam.pt')

for f in ['smoking_cbam.pt','smoking_comparison.png','smoking_balance.png']:
    if os.path.exists(f): files.download(f); print(f'✅ {f}')